In [48]:
query="How does approximate nearest neighbor search work?"



In [49]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

In [50]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [51]:
v = model.encode(query)

In [52]:
v[0] #Q1

np.float32(-0.020582044)

In [53]:
indix=0
for i,doc in enumerate(documents):
    if doc['filename']=='02-vector-search/lessons/07-sqlitesearch-vector.md':
        # print (doc)
        index=i
content=documents[i]['content']
ch02_07_enc=model.encode(content)
v.dot(ch02_07_enc) #Q2

np.float32(0.06656501)

In [54]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)


In [55]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_vectors = model.encode([c['content'] for c in batch])
    X.extend(batch_vectors)


  0%|          | 0/6 [00:00<?, ?it/s]

In [56]:
scores = []

for i in range(len(X)):
    score = v.dot(X[i])
    scores.append(score)
chunks[np.argmax(scores)]['filename'] #Q3

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [57]:
from minsearch import VectorSearch
q="What metric do we use to evaluate a search engine?"
qv=model.encode(q)
vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(np.array(X), chunks)

vindex.search(qv, num_results=1) #Q4

[{'start': 0,
  'content': "# Evaluation\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=eC_IcxfxoiQ&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous modules, we built search engines and RAG pipelines.\nWe tried different approaches: keyword search with minsearch, vector\nsearch, agents with function calling. But we never answered the obvious\nquestion of which one is actually better.\n\nWe could try a few queries by hand and see what looks good. That's fine\nfor a quick sanity check, but it doesn't scale, and it doesn't give us a\nnumber to compare. We need a systematic way to tell whether one approach\nbeats another.\n\nThat's what evaluation is for. And it's worth saying up front: of\neverything in this course, evaluation is the part that matters most. It's\nalso the most tedious. But it's the only way to be sure your system\nworks. And it's how you keep it working as you change prompts and swap\nmodels.\n\n## The evaluation setup\n\nFor search evaluation, w

In [58]:
from minsearch import Index
index=Index(text_fields=['content'],keyword_fields=['filename']).fit(chunks)

q="How do I store vectors in PostgreSQL?"
qv=model.encode(q)
index_results=index.search(q)[:5]
vector_results=vindex.search(qv, num_results=5) 
print (vector_results)
vector_files = {r['filename'] for r in vector_results}
index_files = {r['filename'] for r in index_results}
print(vector_files - index_files) #Q5

[{'start': 3000, 'content': 'PGVector:\n\n```python\ndef vec_to_str(vector):\n    return "[" + ",".join(str(x) for x in vector) + "]"\n\nfor doc, vec in tqdm(zip(documents, vectors), total=len(documents)):\n    conn.execute(\n        """\n        INSERT INTO documents (course, section, question, answer, embedding)\n        VALUES (%s, %s, %s, %s, %s::vector)\n        """,\n        (doc["course"], doc["section"], doc["question"], doc["answer"],\n         vec_to_str(vec))\n    )\n\nconn.commit()\n```\n\nWe loop over the documents and insert each one with its embedding. We\nhand Postgres the vector as text, so the `::vector` cast tells it to\nparse that string back into a vector. We call `conn.commit()` to persist\nthe changes.\n\n## Searching with cosine similarity\n\nSearch with a query:\n\n```python\nquery = "I just discovered the course. Can I still join it?"\nquery_vector = model.encode(query)\nquery_str = vec_to_str(query_vector)\n```\n\nSearch for the most similar documents:\n\n```

In [59]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q="How do I give the model access to tools"
qv=model.encode(q)

index_results=index.search(q)[:5]
vector_results=vindex.search(qv, num_results=5) 

results = rrf([vector_results, index_results])





In [60]:
results[0]['filename'] #q6

'01-agentic-rag/lessons/14-agentic-loop.md'